# Tensors & autograd: hands-on (optional)
**LLMs, Animated · Lesson P4.1**

Four short exercises with PyTorch (pre-installed on Colab; a CPU runtime is fine):
1. tensor shapes and reductions,
2. broadcasting, including fixing a silent shape bug,
3. autograd versus the gradients you derived by hand in P2.3,
4. fitting a line with nothing but tensors, `.backward()` and `torch.no_grad()`.

Each exercise has a cell with a `# TODO` and a **check** cell with `assert`s. Solutions are at the very bottom; try first.

In [ ]:
import torch
torch.manual_seed(0)
print("torch", torch.__version__)

## 1 · Shapes, dtypes and reductions
An LLM's hidden states have shape `(B, T, d)`: batch, sequence position, feature.
Create `h`, a tensor of standard-normal numbers with `B = 4`, `T = 6`, `d = 8`. Then compute:
- `per_token_mean`: the mean over the feature dimension, **keeping** that dimension (shape `(4, 6, 1)`),
- `per_sequence_sum`: the sum over the sequence dimension (shape `(4, 8)`),
- `ids`: the integer tensor `[[464, 3797, 3332]]` (token ids must be `int64`).

In [ ]:
# TODO
h = None
per_token_mean = None
per_sequence_sum = None
ids = None

**Check 1**

In [ ]:
assert h.shape == (4, 6, 8) and h.dtype == torch.float32
assert per_token_mean.shape == (4, 6, 1)
assert torch.allclose(per_token_mean[2, 3, 0], h[2, 3].mean())
assert per_sequence_sum.shape == (4, 8)
assert torch.allclose(per_sequence_sum[1], h[1].sum(dim=0))
assert ids.dtype == torch.int64 and ids.shape == (1, 3)
print("✓ exercise 1")

## 2 · Broadcasting
**2a.** Predict, *without running anything*, the shape of each result (write `None` if it raises an error). Then the check cell will verify.

**2b.** A bug: `scores` has one row per example, shape `(5, 3)`, and `row_max` should be subtracted from **each row**.
The buggy line below broadcasts the wrong way, or fails. Fix it so that every row's maximum becomes 0.

In [ ]:
# 2a: TODO replace each None with a shape tuple, or leave None if it errors
shape_a = None   # torch.ones(3, 1) + torch.ones(4)
shape_b = None   # torch.ones(8, 1, 5) + torch.ones(3, 1)
shape_c = None   # torch.ones(3, 2) + torch.ones(4)
shape_d = None   # torch.ones(3) + torch.ones(3, 1)

# 2b
scores = torch.randn(5, 3)
row_max = scores.max(dim=1).values        # shape (5,)
# TODO: fix the next line
shifted = scores - row_max

**Check 2**

In [ ]:
def actual(f):
    try:
        return tuple(f().shape)
    except RuntimeError:
        return None
assert shape_a == actual(lambda: torch.ones(3, 1) + torch.ones(4))
assert shape_b == actual(lambda: torch.ones(8, 1, 5) + torch.ones(3, 1))
assert shape_c == actual(lambda: torch.ones(3, 2) + torch.ones(4))
assert shape_d == actual(lambda: torch.ones(3) + torch.ones(3, 1)), "(3,) + (3, 1) quietly makes a (3, 3) matrix"
assert shifted.shape == (5, 3)
assert torch.allclose(shifted.max(dim=1).values, torch.zeros(5))
print("✓ exercise 2")

## 3 · Autograd versus your hand-derived gradients
The tiny network from P2.3: $z = w\,x$, $h = \mathrm{relu}(z)$, $\hat y = v\,h$, $L = (\hat y - y)^2$.
Build it with tensors so that autograd computes `w.grad` and `v.grad`, and compare with the hand-derived values ($\partial L/\partial w = -1.5$, $\partial L/\partial v = -6$).

In [ ]:
x, y = torch.tensor(1.5), torch.tensor(2.5)
# TODO: create w = 2.0 and v = 0.5 as leaf tensors that require grad,
# compute L with torch.relu, and call backward
w = None
v = None
L = None

**Check 3.** Also note what happens if you call the forward and backward again without resetting.

In [ ]:
assert w.is_leaf and w.requires_grad
assert abs(L.item() - 1.0) < 1e-6
assert abs(w.grad.item() - (-1.5)) < 1e-6 and abs(v.grad.item() - (-6.0)) < 1e-6
assert L.grad_fn is not None and type(L.grad_fn).__name__ == "PowBackward0"

L2 = (v * torch.relu(w * x) - y) ** 2
L2.backward()
print("after a second backward, w.grad =", w.grad.item(), "(accumulated: -1.5 + -1.5)")
assert abs(w.grad.item() - (-3.0)) < 1e-6
w.grad = None; v.grad = None
print("✓ exercise 3")

## 4 · Fit a line with raw autograd
No `nn`, no optimizer: just tensors. Fit $\hat y = w\,x + b$ to noisy data from $y = 3x - 1$ with full-batch gradient descent and MSE.
Each step: compute the loss, `backward()`, then update **inside `torch.no_grad()`**, then reset the gradients.

In [ ]:
X = torch.linspace(-2, 2, 100)
Y = 3 * X - 1 + 0.2 * torch.randn(100)

In [ ]:
def fit(steps=200, lr=0.1):
    w = torch.zeros((), requires_grad=True)
    b = torch.zeros((), requires_grad=True)
    losses = []
    for _ in range(steps):
        loss = ((w * X + b - Y) ** 2).mean()
        # TODO: backward; update w and b inside torch.no_grad(); reset their .grad to None
        raise NotImplementedError("Exercise 4")
        losses.append(loss.item())
    return w, b, losses

**Check 4**

In [ ]:
w, b, losses = fit()
print(f"w = {w.item():.3f}, b = {b.item():.3f}, final loss = {losses[-1]:.4f}")
assert abs(w.item() - 3) < 0.1 and abs(b.item() + 1) < 0.1
assert losses[-1] < 0.06 and isinstance(losses[-1], float)
assert w.requires_grad and w.is_leaf, "the update must not turn w into a non-leaf (use no_grad and in-place -=)"
print("✓ exercise 4")

## Experiment
1. In `fit`, delete the two lines that reset `.grad`. Run it with `lr=0.1`. What happens to the loss, and why? *(Hint: P4.1, "gradients accumulate".)*
2. Replace `w -= lr * w.grad` with `w = w - lr * w.grad` (still inside `no_grad`). Why does the next iteration fail or stop learning? *(Hint: is the new `w` a leaf that requires grad?)*
3. Try `lr=1.1`. Compare with the learning-rate regimes from P2.2.

In [ ]:
# scratch space: e.g. watch accumulation directly
t = torch.tensor(1.0, requires_grad=True)
for i in range(3):
    (t * 5).backward()
    print("after backward", i + 1, "t.grad =", t.grad.item())

## Recap
- Tensors have a **shape**, **dtype** and **device**; reductions take a `dim`.
- Broadcasting aligns shapes from the right; 1s stretch. Silent broadcasts are a classic bug.
- `requires_grad=True` + operations → a recorded graph; `.backward()` fills the leaves' `.grad`.
- Gradients **accumulate**: reset them every step. Do updates inside `torch.no_grad()`.

Back to the course and hit **Mark complete** to add the review cards.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
h = torch.randn(4, 6, 8)
per_token_mean = h.mean(dim=-1, keepdim=True)
per_sequence_sum = h.sum(dim=1)
ids = torch.tensor([[464, 3797, 3332]])

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
shape_a = (3, 4)
shape_b = (8, 3, 5)
shape_c = None
shape_d = (3, 3)

scores = torch.randn(5, 3)
row_max = scores.max(dim=1).values
shifted = scores - row_max[:, None]       # or row_max.unsqueeze(1), or max(..., keepdim=True)

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
x, y = torch.tensor(1.5), torch.tensor(2.5)
w = torch.tensor(2.0, requires_grad=True)
v = torch.tensor(0.5, requires_grad=True)
L = (v * torch.relu(w * x) - y) ** 2
L.backward()

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def fit(steps=200, lr=0.1):
    w = torch.zeros((), requires_grad=True)
    b = torch.zeros((), requires_grad=True)
    losses = []
    for _ in range(steps):
        loss = ((w * X + b - Y) ** 2).mean()
        loss.backward()
        with torch.no_grad():
            w -= lr * w.grad
            b -= lr * b.grad
        w.grad = None
        b.grad = None
        losses.append(loss.item())
    return w, b, losses